In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))


from validators.schema_validation import SchemaValidationRule
from validators.data_integrity_validation import DataIntegrityRule
from validators.business_rule_validation import BusinessRuleValidation
from transformers.data_depulication import DataDeduplicator
from transformers.data_standardization import DataStandardization
from transformers.data_loader import SilverLoader
from models.schemas import TableSchema, ColumnSchema
from config.database import engine
import nbformat
from nbclient import NotebookClient

In [3]:
customer_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_customers"
)

geolocation_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_geolocation"
)

sellers_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_sellers"
)

products_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_products"
)

order_payments_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_order_payments"
)

order_reviews_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_order_reviews"
)

orders_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_orders"
)

order_items_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\olist_order_items"
)

product_category_translation_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\product_category_name"
)

customer_df = pd.read_parquet(customer_path)
geolocation_df = pd.read_parquet(geolocation_path)
sellers_df = pd.read_parquet(sellers_path)
products_df = pd.read_parquet(products_path)
order_payments_df = pd.read_parquet(order_payments_path)
order_reviews_df = pd.read_parquet(order_reviews_path)
orders_df = pd.read_parquet(orders_path)
order_items_df = pd.read_parquet(order_items_path)
product_category_translation_df = pd.read_parquet(
    product_category_translation_path
)

In [51]:
customer_schema = TableSchema(
    table_name="customers",
    columns=[
        ColumnSchema("customer_id", critical=True),
        ColumnSchema("customer_unique_id", critical=True),
        ColumnSchema("customer_zip_code_prefix", critical=True),
        ColumnSchema("customer_city"),
        ColumnSchema("customer_state"),
    ],
    primary_key = ["customer_id"]
)


geolocation_schema = TableSchema(
    table_name="geolocation",
    columns=[
        ColumnSchema("geolocation_zip_code_prefix", critical=True),
        ColumnSchema("geolocation_lat", critical=True),
        ColumnSchema("geolocation_lng", critical=True),
        ColumnSchema("geolocation_city", critical=True, standardizable = True),
        ColumnSchema("geolocation_state", critical=True),
    ],
    primary_key = ["geolocation_zip_code_prefix"]
)

sellers_schema = TableSchema(
    table_name="sellers",
    columns=[
        ColumnSchema("seller_id", critical=True),
        ColumnSchema("seller_zip_code_prefix", critical=True),
        ColumnSchema("seller_city"),
        ColumnSchema("seller_state"),
    ],
    primary_key = ["seller_id"]
)

products_schema = TableSchema(
    table_name="products",
    columns=[
        ColumnSchema("product_id", critical=True),
        ColumnSchema("product_category_name", standardizable = True),
        ColumnSchema("product_name_lenght"),
        ColumnSchema("product_description_lenght"),
        ColumnSchema("product_photos_qty"),
        ColumnSchema("product_weight_g"),
        ColumnSchema("product_length_cm"),
        ColumnSchema("product_height_cm"),
        ColumnSchema("product_width_cm"),
    ],
    primary_key = ["product_id"]
)

order_payments_schema = TableSchema(
    table_name="order_payments",
    columns=[
        ColumnSchema("order_id", critical=True),
        ColumnSchema("payment_sequential", critical=True),
        ColumnSchema("payment_type", critical=True, standardizable = True),
        ColumnSchema("payment_installments", critical=True),
        ColumnSchema("payment_value", critical=True),
    ],
    primary_key = ["order_id", "payment_sequential"]
)

order_reviews_schema = TableSchema(
    table_name="order_reviews",
    columns=[
        ColumnSchema("review_id", critical=True),
        ColumnSchema("order_id", critical=True),
        ColumnSchema("review_score", critical=True),
        ColumnSchema("review_comment_title"),
        ColumnSchema("review_comment_message"),
        ColumnSchema("review_creation_date"),
        ColumnSchema("review_answer_timestamp"),
    ],
    primary_key = ["review_id", "order_id"]
)

orders_schema = TableSchema(
    table_name="orders",
    columns=[
        ColumnSchema("order_id", critical=True),
        ColumnSchema("customer_id", critical=True),
        ColumnSchema("order_status", critical=True, standardizable = True),
        ColumnSchema("order_purchase_timestamp", critical=True),
        ColumnSchema("order_approved_at"),
        ColumnSchema("order_delivered_carrier_date"),
        ColumnSchema("order_delivered_customer_date"),
        ColumnSchema("order_estimated_delivery_date"),
    ],
    primary_key = ["order_id"]
)

order_items_schema = TableSchema(
    table_name="order_items",
    columns=[
        ColumnSchema("order_id", critical=True),
        ColumnSchema("order_item_id", critical=True),
        ColumnSchema("product_id", critical=True),
        ColumnSchema("seller_id", critical=True),
        ColumnSchema("shipping_limit_date", critical=True),
        ColumnSchema("price", critical=True),
        ColumnSchema("freight_value", critical=True),
    ],
    primary_key = ["order_id", "order_item_id"]
)


product_category_translation_schema = TableSchema(
    table_name="product_category_translation",
    columns=[
        ColumnSchema("product_category_name", critical=True, standardizable = True),
        ColumnSchema("product_category_name_english", critical=True, standardizable = True),
    ],
    primary_key = ["product_category_name"]
)

table_configs = {
    "customers": (customer_df, customer_schema),
    "geolocation": (geolocation_df, geolocation_schema),
    "sellers": (sellers_df, sellers_schema),
    "products": (products_df, products_schema),
    "order_payments": (order_payments_df, order_payments_schema),
    "order_reviews": (order_reviews_df, order_reviews_schema),
    "orders": (orders_df, orders_schema),
    "order_items": (order_items_df, order_items_schema),
    "product_category_translation": (
        product_category_translation_df,
        product_category_translation_schema,
    ),
}

## Phase 1: Schema Validation

In [52]:
rule = SchemaValidationRule()

for table_name, (df, schema) in table_configs.items():

    result_mc = rule.validate_missing(df, schema)

    if result_mc.has_errors:

        print(f"{table_name} failed validation")
        print(result_mc.critical_missing)

        break

    if result_mc.optional_missing:

        print(f"{table_name} has optional columns missing")
        print(result_mc.optional_missing)

    result_uc = rule.validate_unexpected(df, schema)

    if result_uc:
        print(
            f"[WARNING] {table_name}: Unexpected columns detected: "
            f"{', '.join(result_uc)}"
        )
        print(
            "           Review schema evolution before loading into Silver."
        )
    
    print(f"{table_name} schema processed.")

customers schema processed.
geolocation schema processed.
sellers schema processed.
products schema processed.
order_payments schema processed.
order_reviews schema processed.
orders schema processed.
order_items schema processed.
product_category_translation schema processed.


## Phase 2: Data Standardization


In [53]:
standardizer = DataStandardization()

for table_name, (df, schema) in table_configs.items():

    df = standardizer.trim_whitespace(df)
    df = standardizer.convert_empty_strings_to_null(df)
    df = standardizer.standardize_text_case(df, schema)
    if not table_name == "order_reviews":
        df = standardizer.remove_duplicate_spaces(df)

    # Update the dataframe back into the dictionary
    table_configs[table_name] = (df, schema)

## Phase 3: Data Integrity Validation

need to think what to do if there is null or duplicates

In [54]:
rule = DataIntegrityRule()
depulicator = DataDeduplicator()


for table_name, (df, schema) in table_configs.items():
    print(f"Currently processing {table_name}......")

#=================================================================================
# PK NULL CHECK
#=================================================================================
    
    null_pk = rule.check_null_pk(df, schema)
    
    print(f"null pk validation for {table_name}: {null_pk["passed"]}")
    # print(null_pk["primary_key"])

    if not null_pk["passed"]:
        # add null handling logic here
        print(null_pk["null_count"])

        # remove the null pk
        null_pk_columns = null_pk["primary_key"]
        df = df.dropna(subset=pk_columns)
        table_configs[table_name] = (df, schema)
        print(f"{null_pk_columns} nulls removed")
        print("")

#=================================================================================
# PK DUPE CHECK
#=================================================================================

    dupe_pk = rule.check_duplicate_pk(df, schema)

    print(f"duplicate pk validation for {table_name}: {dupe_pk["passed"]}")
    # print(dupe_pk["primary_key"])
    # print(dupe_pk["duplicate_count"])
    
    if not dupe_pk["passed"]:
        # add duplication handling logic here
        # print(dupe_pk["duplicate_rows"])
        if table_name == "geolocation":
            df = depulicator.deduplicate_geolocation(df)
            table_configs[table_name] = (df, schema)
            print(f"{table_name} duplicates removed (avg)")
            if df["geolocation_zip_code_prefix"].duplicated().any():
                print("Duplicates still exist.")
            else:
                print("Successfully deduplicated.")
            print("")
        else:
            # remove the null pk
            dupe_pk_columns = dupe_pk["primary_key"]
            df = df.drop_duplicate(subset=dupe_pk_columns)
            table_configs[table_name] = (df, schema)
            print(f"{dupe_pk_columns} duplicates removed")
            print("")
                
#=================================================================================
# CRITICAL NULL CHECK
#=================================================================================
    
    null_critical = rule.check_null_critical(df, schema)

    print(f"null values validation for {table_name}: {null_critical["passed"]}")
    if not null_critical["passed"]:
        print(null_critical["null_columns"])
        print(null_critical["null_counts"])
        null_critical_column = null_critical["critical_columns"]
        df = df.dropna(subset=null_critical_column)
        table_configs[table_name] = (df, schema)
        print(f"{null_critical_column} nulls removed")
        print("")

#=================================================================================
# DUPE CHECK
#=================================================================================
    dupe_values = rule.check_duplicate_rows(df)

    print(f"duplicated values validation for {table_name}: {dupe_values["passed"]}")

    if not dupe_values["passed"]:
        df = df.drop_duplicate()
        table_configs[table_name] = (df, schema)
        print(f"{dupe_values["duplicate_counts"]} duplciates removed.")


Currently processing customers......
null pk validation for customers: True
duplicate pk validation for customers: True
null values validation for customers: True
duplicated values validation for customers: True
Currently processing geolocation......
null pk validation for geolocation: True
duplicate pk validation for geolocation: False
geolocation duplicates removed (avg)
Successfully deduplicated.

null values validation for geolocation: True
duplicated values validation for geolocation: True
Currently processing sellers......
null pk validation for sellers: True
duplicate pk validation for sellers: True
null values validation for sellers: True
duplicated values validation for sellers: True
Currently processing products......
null pk validation for products: True
duplicate pk validation for products: True
null values validation for products: True
duplicated values validation for products: True
Currently processing order_payments......
null pk validation for order_payments: True
dupli

## Phase 4: Business Rule Validation


In [55]:
validator = BusinessRuleValidation()

validation_rules = {
    "geolocation": [
        validator.validate_state_codes,
    ],
    "order_items": [
        validator.validate_payment_amount,
    ],
    "order_payments": [
        validator.validate_payment_amount,
    ],
    "order_reviews": [
        validator.validate_review_score,
    ],
    "orders": [
        validator.validate_delivery_dates,
    ],
}


for table_name, (df, schema) in table_configs.items():

    if table_name not in validation_rules:
        continue

    for rule in validation_rules[table_name]:

        result = rule(df)

        # log_step(
        #     conn,
        #     run_id,
        #     table_name,
        #     "Business Validation",
        #     rule.__name__,
        #     "SUCCESS" if result["passed"] else "FAILED",
        # )

        if not result["passed"]:
            print(f"{table_name} failed {rule.__name__}")
            if table_name == "geolocation":
                print(f"Invalid rows: {len(result['invalid_rows'])}")

            # Remove invalid rows
            df = df.drop(index=result["invalid_rows"])
        if result["passed"]:
            print(f"{table_name} passed {rule.__name__}")

    # Save the cleaned DataFrame back
    table_configs[table_name] = (df, schema)

geolocation passed validate_state_codes
order_payments passed validate_payment_amount
order_reviews passed validate_review_score
orders passed validate_delivery_dates
order_items passed validate_payment_amount


## Phase 5: Data Transformation

First, we need to translate the categories before moving into gold (don't put tranlation table in gold)

1. create the silver schema, then move columns from bronze to silver


In [56]:
customer_silver_schema = TableSchema(
    table_name="customers",
    columns=[
        ColumnSchema("customer_id", critical=True),
        ColumnSchema("customer_unique_id", critical=True),
        ColumnSchema("customer_zip_code_prefix", critical=True)
    ],
    primary_key = ["customer_id"]
)


geolocation_silver_schema = TableSchema(
    table_name="geolocation",
    columns=[
        ColumnSchema("geolocation_zip_code_prefix", critical=True),
        ColumnSchema("geolocation_lat", critical=True),
        ColumnSchema("geolocation_lng", critical=True),
        ColumnSchema("geolocation_city", critical=True, standardizable = True),
        ColumnSchema("geolocation_state", critical=True),
    ],
    primary_key = ["geolocation_zip_code_prefix"]
)

sellers_silver_schema = TableSchema(
    table_name="sellers",
    columns=[
        ColumnSchema("seller_id", critical=True),
        ColumnSchema("seller_zip_code_prefix", critical=True)
    ],
    primary_key = ["seller_id"]
)

products_silver_schema = TableSchema(
    table_name="products",
    columns=[
        ColumnSchema("product_id", critical=True),
        ColumnSchema("product_category_name_english", standardizable = True),
        ColumnSchema("product_name_lenght"),
        ColumnSchema("product_description_lenght"),
        ColumnSchema("product_photos_qty"),
        ColumnSchema("product_weight_g"),
        ColumnSchema("product_length_cm"),
        ColumnSchema("product_height_cm"),
        ColumnSchema("product_width_cm"),
    ],
    primary_key = ["product_id"]
)

order_payments_silver_schema = TableSchema(
    table_name="order_payments",
    columns=[
        ColumnSchema("order_id", critical=True),
        ColumnSchema("payment_sequential", critical=True),
        ColumnSchema("payment_type", critical=True, standardizable = True),
        ColumnSchema("payment_installments", critical=True),
        ColumnSchema("payment_value", critical=True),
    ],
    primary_key = ["order_id", "payment_sequential"]
)

order_reviews_silver_schema = TableSchema(
    table_name="order_reviews",
    columns=[
        ColumnSchema("review_id", critical=True),
        ColumnSchema("order_id", critical=True),
        ColumnSchema("review_score", critical=True),
        ColumnSchema("review_comment_title"),
        ColumnSchema("review_comment_message"),
        ColumnSchema("review_creation_date"),
        ColumnSchema("review_answer_timestamp"),
    ],
    primary_key = ["review_id", "order_id"]
)

orders_silver_schema = TableSchema(
    table_name="orders",
    columns=[
        ColumnSchema("order_id", critical=True),
        ColumnSchema("customer_id", critical=True),
        ColumnSchema("order_status", critical=True, standardizable = True),
        ColumnSchema("order_purchase_timestamp", critical=True),
        ColumnSchema("order_approved_at"),
        ColumnSchema("order_delivered_carrier_date"),
        ColumnSchema("order_delivered_customer_date"),
        ColumnSchema("order_estimated_delivery_date"),
    ],
    primary_key = ["order_id"]
)

order_items_silver_schema = TableSchema(
    table_name="order_items",
    columns=[
        ColumnSchema("order_id", critical=True),
        ColumnSchema("order_item_id", critical=True),
        ColumnSchema("product_id", critical=True),
        ColumnSchema("seller_id", critical=True),
        ColumnSchema("shipping_limit_date", critical=True),
        ColumnSchema("price", critical=True),
        ColumnSchema("freight_value", critical=True),
    ],
    primary_key = ["order_id", "order_item_id"]
)

# products_df = (
#     products_df
#     .merge(
#         translation_df,
#         on="product_category_name",
#         how="left"
#     )
#     .drop(columns=["product_category_name"])
# )

table_silver_configs = {
    "customers": (customer_df, customer_silver_schema),
    "geolocation": (geolocation_df, geolocation_silver_schema),
    "sellers": (sellers_df, sellers_silver_schema),
    "products": (products_df, products_silver_schema),
    "order_payments": (order_payments_df, order_payments_silver_schema),
    "order_reviews": (order_reviews_df, order_reviews_silver_schema),
    "orders": (orders_df, orders_silver_schema),
    "order_items": (order_items_df, order_items_silver_schema),
}

In [57]:
transformer = SilverLoader(engine, table_configs, table_silver_configs)


load_sequence = [
    "customers",
    "sellers",
    "products",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "geolocation",
]

transformer.run(load_sequence)

Loaded 99441 rows into customers.
Loaded 3095 rows into sellers.
Loaded 32951 rows into products.
Loaded 99441 rows into orders.
Loaded 112650 rows into order_items.
Loaded 103886 rows into order_payments.
Loaded 99224 rows into order_reviews.
Loaded 19015 rows into geolocation.


In [ ]:


def run_notebook(path):
    with open(path) as f:
        nb = nbformat.read(f, as_version=4)

    client = NotebookClient(nb)
    client.execute()

run_notebook("silver_to_gold.ipynb")